# v0.22.0 — Native typed relations

In SurrealDB a graph edge is a record of its own: it has an `id`, the two records it links
(`in` and `out`) and whatever data you put on it. Since v0.22.0 an edge can be a **model**:

* `BaseSurrealEdge` declares the two endpoints, `in_` (stored in the column `in`, a Python
  keyword) and `out`. A subclass declares only the edge's data, which Pydantic validates.
* Edges are created with SurrealDB's `INSERT RELATION`, through the official SDK's
  `insert_relation()`: `relate(EdgeModel, …)`, `save()` and `bulk_create()`. They are real
  relations, so `->purchased->Product`, `traverse()` and `get_related()` see them.
* Edges are read, filtered and watched like any model: `filter(in_=alice)`, `get_or_create()`,
  `live()`, `changes()`.
* The graph helpers take the full SurrealDB-ORM's parameters: `reverse=`, keyword edge fields,
  `tx=` on every write helper, a list of edges, `get_related(direction="both")`.

The contract is the same on SurrealDB 2.x and 3.x. Where the two lines differ on the server (a
duplicate edge id, section 10), the ORM behaves the same on both.

## 1. Connect

In [1]:
import contextlib
import os

from surreal_orm_lite import SurrealDBConnectionManager

HOST = os.environ.get("SURREALDB_HOST", "localhost")
PORT = os.environ.get("SURREALDB_PORT", "8000")

# WebSocket: on SurrealDB 3.x, transactions are then interactive (section 13).
SurrealDBConnectionManager.set_connection(
    url=f"ws://{HOST}:{PORT}/rpc",
    user="root",
    password="root",
    namespace="examples",
    database="examples",
)
print("Connection configured:", SurrealDBConnectionManager.is_connection_set())

Connection configured: True


## 2. Models: two nodes and two edges

`User` and `Product` are ordinary models. `Purchased` and `Follows` are edges:
`BaseSurrealEdge` brings `id`, `in_` and `out`, the subclass adds the edge's data.

* `in_` and `out` hold a `RecordID`. Each accepts a saved model instance, a `RecordID` or a
  `"table:id"` string; anything else (an unsaved instance, `"nocolon"`) is a `ValidationError`.
* `table_name` (new in v0.22.0, for every model) maps a model to its table: here the lower-case
  `purchased` and `follows` that a string `relate("purchased", …)` writes. Without it the table is
  the class name.

In [2]:
from pydantic import Field, ValidationError
from surrealdb import RecordID

from surreal_orm_lite import BaseSurrealEdge, BaseSurrealModel, SurrealConfigDict, SurrealDbError


class User(BaseSurrealModel):
    id: str | None = None
    name: str


class Product(BaseSurrealModel):
    id: str | None = None
    name: str


class Purchased(BaseSurrealEdge):
    model_config = SurrealConfigDict(table_name="purchased")

    quantity: int = Field(ge=1)
    price: float


class Follows(BaseSurrealEdge):
    model_config = SurrealConfigDict(table_name="follows")

    weight: int = 1


print("Purchased fields:", list(Purchased.model_fields))
print("tables:", Purchased.get_table_name(), Follows.get_table_name())
print("in_ is stored as:", Purchased.to_db_field("in_"))

Purchased fields: ['id', 'in_', 'out', 'quantity', 'price']
tables: purchased follows
in_ is stored as: in


## 3. Reset (idempotent)

The notebook is re-executed in place, so it starts by dropping its tables. `REMOVE TABLE` also
drops a table's indexes, such as the `likes_pair` index of section 9. On SurrealDB 3.x removing a
table that does not exist raises, hence the suppressed errors.

In [3]:
client = await SurrealDBConnectionManager.get_client()

DEMO_TABLES = ["purchased", "follows", "likes", "User", "Product"]


async def reset() -> None:
    """Drop every table of this notebook, whether or not it exists."""
    with contextlib.suppress(Exception):
        await client.query("REMOVE INDEX likes_pair ON likes;", {})
    for table in DEMO_TABLES:
        with contextlib.suppress(Exception):
            await client.query(f"REMOVE TABLE {table};", {})


await reset()
print("Demo tables reset")

Demo tables reset


## 4. Nodes

In [4]:
alice = await User(id="alice", name="Alice").save()
bob = await User(id="bob", name="Bob").save()
carol = await User(id="carol", name="Carol").save()
widget = await Product(id="widget", name="Widget").save()
gadget = await Product(id="gadget", name="Gadget").save()
print(await User.objects().count(), "users,", await Product.objects().count(), "products")

3 users, 2 products


## 5. `relate(EdgeModel, …)`: a typed edge back

With an edge class instead of a table name, `relate()` validates the edge's data, creates the
edge with `INSERT RELATION` and returns the edge model, its `id` set by the server. The edge's
fields go as keyword arguments (the full SurrealDB-ORM's spelling) or in `data=`, or both; a
field given both ways is a `TypeError`, raised before anything is sent.

In [5]:
edge = await alice.relate(Purchased, widget, quantity=2, price=9.5)
print(type(edge).__name__, "| id:", type(edge.id).__name__, "in table", edge.id.table_name)
print("in_ =", edge.in_, "| out =", edge.out, "| quantity =", edge.quantity, "| price =", edge.price)

edge = await alice.relate(Purchased, gadget, data={"quantity": 1, "price": 4.0})
print("with data=:", edge.in_, "->", edge.out, "| quantity =", edge.quantity)

try:
    await alice.relate(Purchased, gadget, data={"quantity": 1, "price": 4.0}, quantity=3)
except TypeError as exc:
    print("TypeError:", exc)

Purchased | id: RecordID in table purchased
in_ = User:alice | out = Product:widget | quantity = 2 | price = 9.5
with data=: User:alice -> Product:gadget | quantity = 1
TypeError: relate() got edge field(s) ['quantity'] both in data= and as keyword arguments; pass each one once.


## 6. Validation happens before anything is sent

`Purchased.quantity` is `Field(ge=1)`: `quantity=0` fails in Pydantic, so the server never sees
the edge.

In [6]:
before = await Purchased.objects().count()
try:
    await alice.relate(Purchased, widget, quantity=0, price=9.5)
except ValidationError as exc:
    error = exc.errors()[0]
    print("ValidationError:", error["loc"], "-", error["msg"])
print("purchased edges before:", before, "| after:", await Purchased.objects().count())

ValidationError: ('quantity',) - Input should be greater than or equal to 1
purchased edges before: 2 | after: 2


## 7. `save()`, `bulk_create()` and reading edges

An edge instance saves like any model; `bulk_create()` sends the whole batch in one
`insert_relation()` call. `Purchased.model_validate({"in": …})` is the type-checker-clean way to
build an edge from model instances (a type checker expects a `RecordID` for `in_`).

The edges are then read three ways: a graph path, `get_related()` with the edge class, and the
edge model's own `objects()`. On an endpoint, a filter value may be a model instance, a
`RecordID` or a `"table:id"` string: all three find the same edges.

In [7]:
edge = await Purchased(in_=RecordID("User", "bob"), out=RecordID("Product", "widget"), quantity=1, price=9.5).save()
print("save():", type(edge.id).__name__, "|", edge.in_, "->", edge.out)

edges = await Purchased.objects().bulk_create(
    [
        Purchased.model_validate({"in": bob, "out": gadget, "quantity": 2, "price": 4.0}),
        Purchased.model_validate({"in": carol, "out": widget, "quantity": 3, "price": 9.0}),
        Purchased.model_validate({"in": carol, "out": gadget, "quantity": 1, "price": 4.0}),
    ]
)
print("bulk_create():", len(edges), "edges, every one with an id:", all(e.id for e in edges))
print("purchased edges:", await Purchased.objects().count())

save(): RecordID | User:bob -> Product:widget
bulk_create(): 3 edges, every one with an id: True
purchased edges: 6


In [8]:
path = await alice.traverse("->purchased->Product")
print("->purchased->Product:", sorted(str(record_id) for record_id in path))

products = await alice.get_related(Purchased, model_class=Product)
print("get_related(Purchased, model_class=Product):", sorted(p.name for p in products))

by_instance = await Purchased.objects().filter(in_=alice).order_by("quantity").exec()
by_string = await Purchased.objects().filter(in_="User:alice").order_by("quantity").exec()
print("filter(in_=alice):       ", [(str(e.out), e.quantity) for e in by_instance])
print("filter(in_='User:alice'):", [(str(e.out), e.quantity) for e in by_string])
print("same edges:", [str(e.id) for e in by_instance] == [str(e.id) for e in by_string])

->purchased->Product: ['Product:gadget', 'Product:widget']
get_related(Purchased, model_class=Product): ['Gadget', 'Widget']
filter(in_=alice):        [('Product:gadget', 1), ('Product:widget', 2)]
filter(in_='User:alice'): [('Product:gadget', 1), ('Product:widget', 2)]
same edges: True


## 8. Direction: `reverse=True` and `direction="both"`

`reverse=True` relates `target -> edge -> self`, as in the full SurrealDB-ORM: below,
`bob.relate(Follows, alice, reverse=True)` creates `alice -> follows -> bob`.

`get_related()` without `model_class` returns the neighbours' record ids. `direction="both"`
returns the out-neighbours, then the in-neighbours; the order inside each group is unspecified
(it differs between server versions), so the lists are printed sorted.

In [9]:
follow = await bob.relate(Follows, alice, reverse=True)
print("in_ =", follow.in_, "| out =", follow.out)

await carol.relate(Follows, alice, weight=2)  # carol -> follows -> alice

outgoing = await alice.get_related(Follows, direction="out")
incoming = await alice.get_related(Follows, direction="in")
either = await alice.get_related(Follows, direction="both")
print("out: ", sorted(map(str, outgoing)))
print("in:  ", sorted(map(str, incoming)))
print("both:", sorted(map(str, either)))

in_ = User:alice | out = User:bob
out:  ['User:bob']
in:   ['User:carol']
both: ['User:bob', 'User:carol']


## 9. Idempotent edges

`get_or_create()` is the convenient idempotent relate: it looks the pair up and creates the edge
only when there is none. Instances, `RecordID`s and `"table:id"` strings converge on one row.

In [10]:
first, created = await Follows.objects().get_or_create(in_=bob, out=carol)
print("get_or_create(in_=bob, out=carol): created =", created)
again, created = await Follows.objects().get_or_create(in_="User:bob", out="User:carol")
print("get_or_create(in_='User:bob', out='User:carol'): created =", created)
print("same edge:", again.get_raw_id() == first.get_raw_id())
print("bob -> carol edges:", await Follows.objects().filter(in_=bob, out=carol).count())

get_or_create(in_=bob, out=carol): created = True
get_or_create(in_='User:bob', out='User:carol'): created = False
same edge: True
bob -> carol edges: 1


Between its lookup and its create, a concurrent caller can create the same edge: `get_or_create()`
has a race window. SurrealDB's own mechanisms close it on the server, and through the ORM both
raise `SurrealDbError` on 2.x and 3.x alike.

**A deterministic array id**, one id per pair: a second `save()` with that id raises "already
exists" (section 10 explains the guard). For a symmetric relation, sort the pair first. A raw
`RELATE` that names an existing edge id overwrites it on both lines, so create these edges with
`save()` or `bulk_create()`.

In [11]:
a, b = RecordID("User", "alice"), RecordID("User", "bob")
pair_id = RecordID("follows", [a, b])

await Follows(id=pair_id, in_=a, out=b).save()
try:
    await Follows(id=pair_id, in_=a, out=b, weight=5).save()
except SurrealDbError as exc:
    print("SurrealDbError:", exc)

stored = await Follows.objects().filter(id=pair_id).exec()
print("rows with that id:", len(stored), "| weight:", stored[0].weight)

SurrealDbError: There was a problem with the database: Database record `follows:[User:alice, User:bob]` already exists
rows with that id: 1 | weight: 1


**A `UNIQUE` index on `in, out`**: the server refuses a second edge for the same ordered pair,
whatever its id. The index is directional, so the reverse pair is another edge and is allowed.

In [12]:
import re


def hide_random_id(message: object, table: str) -> str:
    """Hide a server-generated record id of `table` in a message, so the output stays stable."""
    return re.sub(rf"`{table}:\w+`", f"`{table}:…`", str(message))


class Likes(BaseSurrealEdge):
    model_config = SurrealConfigDict(table_name="likes")


await client.query("DEFINE INDEX likes_pair ON likes FIELDS in, out UNIQUE;", {})

await alice.relate(Likes, bob)
try:
    await alice.relate(Likes, bob)
except SurrealDbError as exc:
    print("SurrealDbError:", hide_random_id(exc, "likes"))  # some versions name the stored edge

await bob.relate(Likes, alice)  # the reverse pair
print("likes edges:", await Likes.objects().count())

SurrealDbError: There was a problem with the database: Database index `likes_pair` already contains [User:alice, User:bob], with record `likes:…`
likes edges: 2


## 10. An explicit edge id that already exists

A raw `INSERT RELATION` naming an id that already exists does not behave the same on every
server: SurrealDB 2.7.0 and 3.1.5 silently keep the stored edge and return it as if it had been
inserted, while 3.3.3 raises `AlreadyExistsError` (the SurrealDB docs announce a record-exists
error from 3.1.5; 3.1.5 measured otherwise). An edge with an id therefore goes through a guarded
`INSERT RELATION` that raises on every measured version: `SurrealDbError` ("already exists"), and
the stored edge is unchanged.

In [13]:
a, w = RecordID("User", "alice"), RecordID("Product", "widget")
await Purchased(id="p1", in_=a, out=w, quantity=1, price=9.5).save()
try:
    await Purchased(id="p1", in_=a, out=w, quantity=5, price=9.5).save()
except SurrealDbError as exc:
    print("SurrealDbError:", exc)

p1 = await Purchased.objects().get("p1")
print("stored p1: quantity =", p1.quantity)

SurrealDbError: There was a problem with the database: Database record `purchased:p1` already exists
stored p1: quantity = 1


## 11. Endpoints are immutable

SurrealDB ignores a change to an existing edge's endpoints on both lines: an `UPDATE` writes the
other fields and leaves `in`/`out` as they were. The ORM refuses the change instead of letting it
pass silently (`in_` and `out` are frozen fields, and every write that names an endpoint raises
`ValueError` before anything is sent). To move an edge, delete it and relate again.
`edge.model_copy(update={"out": …})` is not checked, since Pydantic skips validation there.

In [14]:
try:
    p1.out = RecordID("Product", "gadget")
except ValidationError as exc:
    print("assignment refused:", exc.errors()[0]["msg"])
try:
    await p1.merge(out=gadget)
except ValueError as exc:
    print("merge(out=…) refused:", exc)

await p1.merge(quantity=3)
p1 = await Purchased.objects().get("p1")
print("after merge(quantity=3):", p1.in_, "->", p1.out, "| quantity =", p1.quantity)

assignment refused: Field is frozen
merge(out=…) refused: merge(): `in`/`out` are the endpoints of edge `Purchased`; SurrealDB ignores a change to them (both lines), so the ORM refuses it — delete the edge and relate again.
after merge(quantity=3): User:alice -> Product:widget | quantity = 3


## 12. A `TYPE RELATION` table, defined by hand

Typed edges work with whatever table exists: the one the first edge created, or a
`TYPE RELATION IN … OUT …` table you define. The server's refusals come back as ORM errors: an
endpoint from the wrong table raises `SurrealDbError` on both lines. (`OVERWRITE` because the
first edge already created `purchased`.) A helper that writes this DDL from the model is
planned for v0.33.0.

In [15]:
await client.query("DEFINE TABLE OVERWRITE purchased TYPE RELATION IN User OUT Product;", {})
try:
    await alice.relate(Purchased, bob, quantity=1, price=1.0)  # User -> User: out must be a Product
except SurrealDbError as exc:
    print("SurrealDbError:", hide_random_id(exc, "purchased"))  # names the refused edge

edge = await alice.relate(Purchased, widget, quantity=1, price=9.5)
print("User -> Product still works:", edge.in_, "->", edge.out)

SurrealDbError: There was a problem with the database: Couldn't coerce value for field `out` of `purchased:…`: Expected `record<Product>` but found `User:bob`
User -> Product still works: User:alice -> Product:widget


## 13. Transactions

`relate()`, `remove_relation()` and `remove_all_relations()` take `tx=`, and edges follow the
transaction's strategy:

* **interactive** (WebSocket + SurrealDB 3.x): the statement runs at once, so `edge.id` is set
  inside the block and `objects(tx=)` reads see the change;
* **buffered** (SurrealDB 2.x, or HTTP): the write is queued and applied at commit, so an
  auto-id edge keeps `id = None` (read it back from the table after the commit), and reads inside
  the block are not available.

This cell runs on both: it prints the strategy it got instead of assuming one.

In [16]:
async with SurrealDBConnectionManager.transaction() as tx:
    print("interactive transaction:", tx.is_interactive)
    follow = await carol.relate(Follows, bob, tx=tx)
    print("inside the block, edge.id set:", follow.id is not None)
print("after the commit, carol -> bob edges:", await Follows.objects().filter(in_=carol, out=bob).count())

interactive transaction: True
inside the block, edge.id set: True
after the commit, carol -> bob edges: 1


A rollback undoes a removal too: the block below removes `alice -> follows -> bob`, then raises.
On an interactive transaction the removal is visible inside the block; on a buffered one it was
never sent. Either way the edges are still there afterwards.

In [17]:
class Abort(Exception):
    """Raised on purpose to roll the transaction back."""


before = await Follows.objects().filter(in_=alice, out=bob).count()
try:
    async with SurrealDBConnectionManager.transaction() as tx:
        await alice.remove_relation(Follows, bob, tx=tx)
        if tx.is_interactive:
            inside = await Follows.objects(tx=tx).filter(in_=alice, out=bob).count()
            print("inside the transaction, alice -> bob edges:", inside)
        else:
            print("buffered transaction: the removal is queued until the commit")
        raise Abort
except Abort:
    pass
after = await Follows.objects().filter(in_=alice, out=bob).count()
print("alice -> bob edges before:", before, "| after the rollback:", after)

inside the transaction, alice -> bob edges: 0
alice -> bob edges before: 2 | after the rollback: 2


## 14. Cleanup

`remove_all_relations()` takes one edge or a list, as classes or table names, and
`direction="both"` removes the edges in both directions in one statement.

In [18]:
async def edge_counts() -> dict[str, int]:
    return {m.get_table_name(): await m.objects().count() for m in (Purchased, Follows, Likes)}


print("before:", await edge_counts())
for node in (alice, bob, carol, widget, gadget):
    await node.remove_all_relations([Purchased, Follows, "likes"], direction="both")
print("after: ", await edge_counts())

await reset()
await SurrealDBConnectionManager.close_connection()
print("Done")

before: {'purchased': 8, 'follows': 5, 'likes': 2}


after:  {'purchased': 0, 'follows': 0, 'likes': 0}


Done
